# P1-28v · Experiment E3: training on radiologist breast masks (VinDr-Mammo, 5-fold CV by exam) — session D
**Runtime: T4 GPU.** Run copies A and B at the same time; they share the 100 runs (4 variants x 5 seeds x 5 folds) through lock files.
Resumable: after a disconnect, *Disconnect and delete runtime*, reconnect and **Run all**; an interrupted run continues from its last stored epoch (every 20 epochs). Run P1-26b, P1-27c and P1-29 first.
Read only: `QSEG_FINAL`, `P1_v2/e3`, `e3_refs`, `e3_csaws`, `mias_ext`, `revision_checks`. Writes only `MyDrive/P1_v2/e3v_runs/`.
The prespecified text (cell 1) is stored with a UTC time stamp before the first run. Target region: **breast**.

In [ ]:
# 1 · Settings, Drive, protocol record
SESSION = 'D'
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, sys, json, time, shutil, random, gc, hashlib
import numpy as np, pandas as pd
ROOT = '/content/drive/MyDrive/QSEG_FINAL'          # read only
OUT  = '/content/drive/MyDrive/P1_v2'
E3 = f'{OUT}/e3'; REFS = f'{OUT}/e3_refs'; CS = f'{OUT}/e3_csaws'; MIAS = f'{OUT}/mias_ext'; RC = f'{OUT}/revision_checks'
RUNS_DIR = f'{OUT}/e3v_runs'
LOCAL1 = '/content/qseg_local'; LOCAL3 = '/content/e3_local'; TMP = '/content/e3_tmp'
for d in (RUNS_DIR, TMP): os.makedirs(d, exist_ok=True)
USE_CSAWS = False     # fixed in the protocol before the first run
CSAWS_EXPERTS = ['annotator_1', 'annotator_2']     # test-set folders of the radiologists (data descriptor: experts 1 and 2 drew the anatomy)
for p, nb_ in ((E3 + '/data/manifest.csv', 'P1-27c'), (RC + '/masks_final/breast', 'P1-26b'), (REFS + '/reader1', 'P1-27c')) + (((CS + '/manifest.csv', 'P1-29'),) if USE_CSAWS else ()):
    assert os.path.exists(p), f'{p} missing: run notebook {nb_} first'
PROTO = """Paper 1 - experiment E3 (training on radiologist breast masks), written before any of these runs was trained.

Question: does the four-qubit circuit change segmentation accuracy relative to parameter-matched controls when the networks are
trained on expert (radiologist) masks of a public full-field digital mammography data set instead of rule-based masks of one private
hospital set?
Training data: VinDr-Mammo (PhysioNet, version 1.0.0; credentialed access under its data use agreement) images that carry the expert
radiologist breast masks of the UEF breast-density data set (Mendeley Data 10.17632/tdx3h2fn9v.5, CC BY-NC 4.0): 596 images of 573
exams. Images: DICOM, MONOCHROME1 inverted, intensity window 1st-99.5th percentile of the non-background pixels, Paper 1 preprocessing
(padding away from the chest wall to a square, 256 x 256). Masks: thresholded at 127, same geometry. Target: breast region without the pectoral muscle; the radiologist masks exclude the pectoral muscle in MLO views (P1-27d: mask coverage of the upper chest-wall corner of MLO views median 0.0, and overlay sheet), so the target is the breast region without the pectoral muscle, and every external reference is used in the same region.
Folds: exams (VinDr study_id) shuffled once with numpy RandomState(42) and dealt round-robin to five folds; for fold k the test set is
fold k, the validation set (early stopping) fold k+1 (mod 5) and the training set the other three folds, so every image is tested
exactly once per seed and no exam contributes to two roles.
Models: HQB-UNet (four-qubit circuit), HQB-cls (Dense(4, tanh)), HQB-trig (20-parameter trigonometric map) and HQB-free (no 4-to-4
map), with skip connections, architectures of E1 unchanged; seeds 0-4 in every fold: 4 x 5 x 5 = 100 runs. Matched initialization:
every variant first builds its own model, then all layers it shares with HQB-UNet are overwritten with the initial weights of the
HQB-UNet built with the same seed (a fingerprint of these initial weights is recorded for every run). The random draws of the
4-to-4 map do not shift any later draw, so decoder initialization and dropout streams are identical across variants. Data order,
augmentation and dropout are fixed by the seed. A run whose validation loss is never finite is recorded as failed and reported.
Training: exactly as E1 (Adam 1e-4, batch 16, BCE + soft Dice, fp32, Paper 1 augmentation, up to 200 epochs, learning rate halved
after 4 epochs without improvement of the training loss (minimum 1e-7), early stopping on the validation loss from epoch 15 with
patience 10, lowest-validation-loss weights evaluated). Threshold 0.5. No hyperparameter is tuned on E3 data.
Primary estimands: HQB-UNet - HQB-cls, HQB-UNet - HQB-trig and HQB-UNet - HQB-free in mean per-image Dice against the radiologist
masks over all 596 images, each image scored by the model of the fold in which it was tested (out-of-fold). Uncertainty: crossed
hierarchical bootstrap over exams and seeds (10,000 replicates, numpy default_rng(0)), 95% percentile interval and two-sided bootstrap
P; Holm over the three contrasts. A contrast is called resolved if its Holm-adjusted P < 0.05. Equivalence within +/-0.01 Dice is
concluded if the 90% interval lies inside +/-0.01. Every fold and every seed is also reported separately.
Secondary, external (no fine-tuning; for every image and seed, Dice is averaged over the five fold models), same three contrasts,
Holm within each reference set, in the target region:
  (1) mini-MIAS, 322 images of 161 women, expert-corrected masks (Verboom et al., Zenodo 10.5281/zenodo.10149914; orientation
      verified visually), women as clusters;
  (2) the 84 hospital test images, manual outlines of reader 1 (all 84) and reader 2 (subset), patients as clusters.
Also reported: the other region of each reference (with/without pectoral muscle), model levels, code replacement at test time
(constant code = mean training code; codes of other training images, five donor draws), finite-shot readout of HQB-UNet (100, 1,000,
10,000 shots), seed-sensitivity checks (independent seed resampling, t interval over seed-level differences, leave one seed out).
All runs are reported; none is discarded. Weights are deleted after evaluation; probability maps (uint8) are kept."""
proto = OUT + '/PROTOCOL_e3_vindr_prespecified.txt'
if not os.path.exists(proto):
    open(proto, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
assert open(proto).read().startswith(PROTO), 'protocol file on Drive differs from this notebook'
print(open(proto).read().splitlines()[-1])
REF = 'breast'          # target region (breast = without pectoral muscle, full = with pectoral muscle)
OTHER = 'full' if REF == 'breast' else 'breast'

In [ ]:
# 2 · Data: E3 (VinDr, local copy), external references
if not os.path.exists(LOCAL3 + '/data/manifest.csv'): shutil.copytree(E3 + '/data', LOCAL3 + '/data')
if not os.path.exists(LOCAL1 + '/data/manifest.csv'):
    shutil.copytree(ROOT + '/results/data', LOCAL1 + '/data', ignore=shutil.ignore_patterns('*.npy', '*.h5'))
sys.path.insert(0, ROOT + '/code')
import tensorflow as tf
from tensorflow.keras import layers, models
assert tf.config.list_physical_devices('GPU'), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
from qseg.config import Config
from qseg.models import conv_block, output_layer, TFCircuitLayer
from qseg.datasets import load_manifest, make_dataset, load_arrays
from qseg.evaluation import per_image_overlap
from qseg.train import bce_dice_loss, soft_dice
cfg = Config(work_dir=LOCAL3); man0 = load_manifest(cfg)
assert len(man0) == 596 and man0.file_id.is_unique, len(man0)
assert man0.groupby('patient_id').fold.nunique().max() == 1, 'an exam is in two folds'
MAN = {k: man0.assign(split=man0[f'split{k}']) for k in range(5)}
for k in range(5):
    assert set(MAN[k].patient_id[MAN[k].split == 'test']).isdisjoint(MAN[k].patient_id[MAN[k].split != 'test'])
X3 = {(k, s): load_arrays(cfg, s, MAN[k]) for k in range(5) for s in ('train', 'test')}
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
# (1) mini-MIAS, expert-corrected masks in the visually verified orientation
mm = pd.read_csv(MIAS + '/manifest.csv'); mm = mm[mm.status == 'ok'].drop_duplicates('file_id').sort_values('file_id').reset_index(drop=True)
assert len(mm) == 322, len(mm)
XM = np.stack([_png(f'{MIAS}/images/{f}.png', False) for f in mm.file_id])
YM = {k: np.stack([_png(f'{RC}/masks_final/{k}/{f}.png', True) for f in mm.file_id]) for k in ('full', 'breast')}
# (2) CSAW-S test set, expert outlines (one mask per annotator)
cid, ANN, YC, cpat, XC = [], [], {}, np.array([]), np.zeros((0, 256, 256, 1), np.float32)
if USE_CSAWS:
    cm = pd.read_csv(CS + '/manifest.csv'); cm = cm[cm.annotator.isin(CSAWS_EXPERTS)]; cid = sorted(cm.file_id.unique()); ANN = sorted(cm.annotator.unique())
    assert len(cid) > 0 and len(ANN) > 0
    XC = np.stack([_png(f'{CS}/images/{f}.png', False) for f in cid])
    cpat = cm.drop_duplicates('file_id').set_index('file_id').patient_id.reindex(cid).values
for a in ANN:
    have = [i for i, f in enumerate(cid) if os.path.exists(f'{CS}/{a}/{REF}/{f}.png')]
    for k in ('full', 'breast'):
        if have: YC[(a, k)] = (have, np.stack([_png(f'{CS}/{a}/{k}/{cid[i]}.png', True) for i in have]))
# (3) hospital test images and the manual outlines of the two readers
cfg1 = Config(work_dir=LOCAL1); H = load_arrays(cfg1, 'test', load_manifest(cfg1)); hid = list(H[2].file_id)
RD = {}
for rd in ('reader1', 'reader2'):
    for k in ('full', 'breast'):
        have = [i for i, f in enumerate(hid) if os.path.exists(f'{REFS}/{rd}/{k}/{f}.png')]
        if have: RD[(rd, k)] = (have, np.stack([_png(f'{REFS}/{rd}/{k}/{hid[i]}.png', True) for i in have]))
assert ('reader1', REF) in RD and len(RD[('reader1', REF)][0]) == 84 and len(hid) == 84, 'reader 1 masks incomplete'
if USE_CSAWS: assert any((a, REF) in YC for a in ANN), 'no CSAW-S expert masks for the target region'
for k in range(5):
    assert (MAN[k].split == 'test').eq(man0.fold == k).all() and (MAN[k].split == 'val').eq(man0.fold == (k + 1) % 5).all()
print({k: len(v[0]) for k, v in X3.items()}, '| val', [int((MAN[k].split == 'val').sum()) for k in range(5)])
print('mini-MIAS', XM.shape, '| CSAW-S', XC.shape, {k: len(v[0]) for k, v in YC.items()}, '| hospital', len(hid),
      {k: len(v[0]) for k, v in RD.items()}, '| TF', tf.__version__)

In [ ]:
# 3 · Models (Paper 1 hybrid architecture; optional removal of the skip connections) and evaluation helpers
class CodeSwitch(layers.Layer):
    """Identity during training; at test time can replace the four-value code by supplied codes."""
    def build(self, input_shape):
        self.on = self.add_weight(shape=(), initializer='zeros', trainable=False, name='on')
        self.buf = self.add_weight(shape=(cfg.batch_size, 4), initializer='zeros', trainable=False, name='buf')
    def call(self, z):
        n = tf.shape(z)[0]
        return z * (1.0 - self.on) + self.buf[:n] * self.on

class TrigLayer(layers.Layer):
    """20-parameter trigonometric 4 -> 4 map: z = tanh(A cos((pi/2) x + phi))."""
    def build(self, input_shape):
        self.A = self.add_weight(shape=(4, 4), initializer='glorot_uniform', name='A')
        self.phi = self.add_weight(shape=(4,), initializer=tf.keras.initializers.RandomUniform(0.0, 2 * np.pi), name='phi')
    def call(self, x):
        return tf.tanh(tf.matmul(tf.cos(np.pi / 2 * x + self.phi), self.A, transpose_b=True))

def build(kind, skips=True, filters=(32, 64, 128, 256, 512), dropouts=(0.1, 0.1, 0.2, 0.2, 0.3)):
    inp = layers.Input((256, 256, 1), name='input_image')
    sk, x = [], inp
    for i, (f, d) in enumerate(zip(filters[:-1], dropouts[:-1])):
        x = conv_block(x, f, d, name=f'enc{i + 1}'); sk.append(x)
        x = layers.MaxPooling2D(2, name=f'pool{i + 1}')(x)
    x = conv_block(x, filters[-1], dropouts[-1], name='bottleneck')
    z = layers.Dense(4, activation='tanh', name='pre_dense')(layers.GlobalAveragePooling2D(name='gap')(x))
    _st = random.getstate()      # E3: the 4-to-4 slot must not shift later random draws (decoder init, dropout seeds)
    if kind == 'quantum':   z = TFCircuitLayer(4, 2, name='quantum_circuit')(z)
    elif kind == 'trig':    z = TrigLayer(name='trig_replacement')(z)
    elif kind == 'classical': z = layers.Dense(4, activation='tanh', name='classical_replacement')(z)
    random.setstate(_st)
    z = CodeSwitch(name='code_switch')(z)
    q = layers.Dense(512, activation='relu', name='post_dense')(z)
    x = layers.Reshape((16, 16, filters[-1]), name='q_reshape')(
        layers.Dense(16 * 16 * filters[-1], activation='relu', name='q_expand')(q))
    for i, (f, s, d) in enumerate(zip(reversed(filters[:-1]), reversed(sk), reversed(dropouts[:-1]))):
        x = layers.UpSampling2D(2, name=f'up{i + 1}')(x)
        if skips: x = layers.Concatenate(name=f'concat{i + 1}')([x, s])
        x = conv_block(x, f, d, name=f'dec{i + 1}')
    return models.Model(inp, output_layer(x), name=f'HQB_{kind}_{"skip" if skips else "noskip"}')

def set_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def predict(model, x, codes=None):
    sw = model.get_layer('code_switch'); out = []
    for a in range(0, len(x), cfg.batch_size):
        xb = x[a:a + cfg.batch_size]
        if codes is None: sw.on.assign(0.0)
        else:
            buf = np.zeros((cfg.batch_size, 4), np.float32); buf[:len(xb)] = codes[a:a + len(xb)]
            sw.buf.assign(buf); sw.on.assign(1.0)
        out.append(model(xb, training=False).numpy()[..., 0])
    sw.on.assign(0.0)
    return np.concatenate(out)

def code_model(model):
    return tf.keras.Model(model.input, model.get_layer('code_switch').input)

def dice(Y, p):
    return per_image_overlap((Y[..., 0] > 0.5).astype(np.uint8), (p >= cfg.threshold).astype(np.uint8))[:, 0]

def shots(z, n, rng):
    """Finite-shot estimate of <Z> per qubit: p(+1) = (1 + z)/2, n shots."""
    p = np.clip((1 + z) / 2, 0, 1)
    return 2 * rng.binomial(n, p) / n - 1

def evaluate(model, kind, x, Y, z_tr):
    cm = code_model(model)
    z = cm.predict(x, batch_size=cfg.batch_size, verbose=0)
    d0 = dice(Y, predict(model, x))
    r = pd.DataFrame({'dice': d0})
    r['dice_check_switch'] = dice(Y, predict(model, x, z))             # must equal dice (sanity check)
    r['drop_const'] = d0 - dice(Y, predict(model, x, np.repeat(z_tr.mean(0)[None], len(x), 0)))
    for k in range(5):
        idx = np.random.default_rng(k).integers(0, len(z_tr), len(x))
        r[f'drop_swap{k}'] = d0 - dice(Y, predict(model, x, z_tr[idx]))
    r['drop_swapavg'] = r[[f'drop_swap{k}' for k in range(5)]].mean(1)
    if kind == 'quantum':
        for n in (100, 1000, 10000):
            ds = [dice(Y, predict(model, x, shots(z, n, np.random.default_rng(100 + k)))) for k in range(5)]
            r[f'dice_shots{n}'] = np.mean(ds, 0)
    r['code_sd'] = float(z.std(0).mean())
    return r


In [ ]:
# 4 · Training and evaluation of all runs (resumable; two sessions share the work through lock files)
KIND = {'hq': 'quantum', 'hc': 'classical', 'hf': 'none', 'ht': 'trig'}
RUNS = [(f'E3v_f{k}_{m}', KIND[m], k, s) for s in range(5) for k in range(5) for m in ('hq', 'hc', 'ht', 'hf')]
LOCK_H = 4
RES_EVERY = 20
def claim(d):
    if os.path.exists(d + '/run.json'): return False
    lk = d + '/.lock'
    if os.path.exists(lk):
        try: who, t = open(lk).read().split(); t = float(t)
        except Exception: who, t = SESSION, 0.0
        if who != SESSION and time.time() - t < LOCK_H * 3600: return False
    os.makedirs(d, exist_ok=True); open(lk, 'w').write(f'{SESSION} {time.time()}')
    time.sleep(60)
    try: return open(lk).read().split()[0] == SESSION
    except Exception: return False

def ext_scores(ph, Ys, prefix, n):
    out = {}
    for key, (idx, Y) in Ys.items():
        col = np.full(n, np.nan)
        if len(idx): col[idx] = dice(Y, ph[idx])
        out[f'{prefix}_{"_".join(key)}'] = col
    return out

def run(name, kind, split, seed):
    d = f'{RUNS_DIR}/{name}_seed{seed}'
    if not claim(d): return
    man = MAN[split]
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.mixed_precision.set_global_policy('float32'); set_seeds(seed)
    tr, _ = make_dataset(cfg, 'train', seed, man); va, _ = make_dataset(cfg, 'val', seed, man)
    model = build(kind, True)
    # matched initialization: all layers shared with the HQB-UNet get the HQB-UNet initial weights of the same seed
    set_seeds(seed); ref = build('quantum', True); refw = {l.name: l.get_weights() for l in ref.layers if l.weights}
    shared = [l for l in model.layers if l.weights and l.name in refw and l.name not in ('quantum_circuit',)]
    for l in shared: l.set_weights(refw[l.name])
    init_check = max(float(np.abs(a - b).max()) for l in shared for a, b in zip(l.get_weights(), refw[l.name]))
    n_shared = int(sum(w.size for l in shared for w in l.get_weights()))
    init_sha = hashlib.sha1(b''.join(w.tobytes() for n_ in sorted(refw) for w in refw[n_])).hexdigest(); del ref, refw; gc.collect()
    set_seeds(seed)
    model.compile(optimizer=tf.keras.optimizers.Adam(cfg.learning_rate), loss=bce_dice_loss, metrics=[soft_dice], jit_compile=False)
    ck = f'{TMP}/{name}_{seed}.weights.h5'
    # --- resumable training: every RES_EVERY epochs the training state is stored in {d}/resume (weights, Adam state, the weights of the
    # best epoch so far, callback counters, epoch); after a disconnect the run continues from the last stored epoch instead of epoch 0
    R = f'{d}/resume'; os.makedirs(R, exist_ok=True)
    es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, start_from_epoch=15)
    rlr = tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=4, min_lr=1e-7)
    mc = tf.keras.callbacks.ModelCheckpoint(ck, monitor='val_loss', save_best_only=True, save_weights_only=True)
    fl = lambda x: None if x is None else float(x)
    def put(arrs, fn):                                           # write locally, then copy to Drive under a temporary name, then rename
        np.savez(f'{TMP}/{fn}', **{f'a{i}': a for i, a in enumerate(arrs)})
        shutil.copy(f'{TMP}/{fn}', f'{R}/{fn}.part'); os.replace(f'{R}/{fn}.part', f'{R}/{fn}')
    def get(fn):
        z = np.load(f'{R}/{fn}'); return [z[f'a{i}'] for i in range(len(z.files))]
    st = json.load(open(f'{R}/state.json')) if os.path.exists(f'{R}/state.json') else None
    init_ep, best_w = 0, None
    if st is not None:                                           # resume
        init_ep = st['epoch']; st['resumed_at'] = st.get('resumed_at', []) + [init_ep]
        best_w = get('best.npz'); model.set_weights(best_w); model.save_weights(ck)      # best epoch so far -> local checkpoint
        model.set_weights(get('model.npz'))
        model.optimizer.build(model.trainable_variables)
        for v, a in zip(model.optimizer.variables, get('opt.npz')): v.assign(a)
        for _ in range(init_ep):                                 # continue the shuffling and augmentation streams
            for _b in tr: pass
        print(f'   {name} seed {seed}: resumed at epoch {init_ep}', flush=True)
    T0 = time.time()
    class Resume(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.mc_last = mc.best
            if st is not None:
                es.best, es.wait, es.best_epoch = st['es_best'], st['es_wait'], st['es_best_epoch']
                rlr.best, rlr.wait, rlr.cooldown_counter = st['rlr_best'], st['rlr_wait'], st['rlr_cooldown']
                mc.best = st['mc_best']; self.mc_last = mc.best; model.optimizer.learning_rate.assign(st['lr'])
        def on_epoch_end(self, epoch, logs=None):
            nonlocal best_w
            if mc.best is not None and mc.best != self.mc_last:  # ModelCheckpoint stored a new best epoch: keep its weights in memory
                best_w, self.mc_last = model.get_weights(), mc.best
            if (epoch + 1) % RES_EVERY or best_w is None: return
            put(model.get_weights(), 'model.npz'); put([v.numpy() for v in model.optimizer.variables], 'opt.npz'); put(best_w, 'best.npz')
            sj = dict(epoch=epoch + 1, es_best=fl(es.best), es_wait=int(es.wait), es_best_epoch=int(es.best_epoch),
                      rlr_best=fl(rlr.best), rlr_wait=int(rlr.wait), rlr_cooldown=int(rlr.cooldown_counter), mc_best=fl(mc.best),
                      lr=float(model.optimizer.learning_rate.numpy()), secs=(st or {}).get('secs', 0.0) + time.time() - T0,
                      resumed_at=(st or {}).get('resumed_at', []))
            json.dump(sj, open(f'{R}/state.json.part', 'w')); os.replace(f'{R}/state.json.part', f'{R}/state.json')
    # Resume comes last: its on_train_begin must run after the other callbacks reset their counters
    cbs = [tf.keras.callbacks.TerminateOnNaN(), mc, es, rlr, tf.keras.callbacks.CSVLogger(d + '/history.csv', append=st is not None), Resume()]
    t0 = time.time(); h = model.fit(tr, validation_data=va, epochs=200, initial_epoch=init_ep, callbacks=cbs, verbose=0)
    secs = time.time() - t0 + (st or {}).get('secs', 0.0)
    if not os.path.exists(ck):
        json.dump(dict(name=name, fold=split, seed=seed, failed='no finite validation loss', epochs_run=len(h.history.get('loss', []))), open(d + '/failed.json', 'w')); return
    model.load_weights(ck)
    z_tr = code_model(model).predict(X3[(split, 'train')][0], batch_size=cfg.batch_size, verbose=0)
    # out-of-fold VinDr test images (primary)
    xt, yt, dft = X3[(split, 'test')]
    te = evaluate(model, kind, xt, yt, z_tr); te.insert(0, 'file_id', dft.file_id.values); te.insert(1, 'patient_id', dft.patient_id.values)
    pt = predict(model, xt)
    # external references (no fine-tuning)
    pm = predict(model, XM)
    mi = pd.DataFrame({'file_id': mm.file_id, 'dice_full': dice(YM['full'], pm), 'dice_breast': dice(YM['breast'], pm)})
    pc = predict(model, XC) if USE_CSAWS else np.zeros((0, 256, 256), np.float32)
    cs = pd.DataFrame({'file_id': cid, 'patient_id': cpat, **ext_scores(pc, YC, 'dice', len(cid))})
    ph = predict(model, H[0])
    ho = pd.DataFrame({'file_id': hid, 'patient_id': H[2].patient_id.values, 'dice_proxy': dice(H[1], ph), **ext_scores(ph, RD, 'dice', len(hid))})
    for t_, f_ in ((te, 'test'), (mi, 'mias'), (cs, 'csaws'), (ho, 'hospital')):
        t_.insert(0, 'run', name); t_.insert(1, 'fold', split); t_.insert(2, 'seed', seed); t_.to_csv(f'{d}/{f_}_per_image.csv', index=False)
    np.savez_compressed(d + '/prob_uint8.npz', test=(pt * 255).round().astype(np.uint8), test_file_id=dft.file_id.values.astype(str),
                        mias=(pm * 255).round().astype(np.uint8), csaws=(pc * 255).round().astype(np.uint8), hospital=(ph * 255).round().astype(np.uint8))
    hist = pd.read_csv(d + '/history.csv').drop_duplicates('epoch', keep='last'); vl = hist['val_loss'].to_numpy()   # all epochs, also across a resume
    lr = hist['learning_rate'] if 'learning_rate' in hist else hist.get('lr', pd.Series([np.nan]))
    csm = np.nanmean(np.stack([cs[f'dice_{a}_{REF}'] for a in ANN if f'dice_{a}_{REF}' in cs]), 0) if USE_CSAWS else np.array([0.0]) * np.nan
    r = dict(name=name, kind=kind, fold=split, seed=seed, target=REF, init_shared_params=n_shared, init_shared_max_abs_diff=init_check, init_sha=init_sha,
             epochs_run=len(vl), best_epoch=int(np.nanargmin(vl)) + 1, final_lr=float(lr.iloc[-1]), minutes=round(secs / 60, 1),
             params=int(model.count_params()), test_dice=float(te.dice.mean()), mias_dice=float(mi[f'dice_{REF}'].mean()),
             csaws_dice=float(np.nanmean(csm)), hospital_reader1=float(np.nanmean(ho[f'dice_reader1_{REF}'])),
             hospital_proxy=float(ho.dice_proxy.mean()), test_drop_const=float(te.drop_const.mean()),
             test_drop_swapavg=float(te.drop_swapavg.mean()), switch_check_max=float(np.abs(te.dice - te.dice_check_switch).max()),
             session=SESSION, tf_version=tf.__version__, resumed_at=(st or {}).get('resumed_at', []))
    json.dump(r, open(d + '/run.json', 'w'), indent=2)
    shutil.rmtree(R, ignore_errors=True); os.remove(ck); del model; tf.keras.backend.clear_session(); gc.collect()
    done = sum(os.path.exists(f'{RUNS_DIR}/{a}_seed{s}/run.json') for a, _, _, s in RUNS)
    print(time.strftime('%H:%M UTC', time.gmtime()), f"| {name} seed {seed}: {r['epochs_run']} ep (best {r['best_epoch']}), {r['minutes']} min | "
          f"VinDr {r['test_dice']:.4f} | mini-MIAS {r['mias_dice']:.4f} | CSAW-S {r['csaws_dice']:.4f} | reader1 {r['hospital_reader1']:.4f} | "
          f"init check {init_check:.1e} | finished {done} / {len(RUNS)}", flush=True)

order = RUNS[:50][::-1] + RUNS[50:][::-1]   # extra session D starts in the middle and moves backward
for args in order:
    run(*args)
left = [a for a in RUNS if not os.path.exists(f'{RUNS_DIR}/{a[0]}_seed{a[3]}/run.json')]
print(f'session {SESSION} has nothing left to start; {len(left)} run(s) still running in the other session')

In [ ]:
# 5 · Overview of finished runs
rows = [json.load(open(f'{RUNS_DIR}/{a}_seed{s}/run.json')) for a, _, _, s in RUNS if os.path.exists(f'{RUNS_DIR}/{a}_seed{s}/run.json')]
ov = pd.DataFrame(rows); ov['variant'] = ov.name.str[-2:]; ov.to_csv(RUNS_DIR + f'/overview_{SESSION}.csv', index=False)
print(len(ov), 'of', len(RUNS), 'runs finished')
print(ov.groupby('variant')[['init_shared_max_abs_diff', 'test_dice', 'mias_dice', 'csaws_dice', 'hospital_reader1', 'hospital_proxy',
                             'test_drop_const', 'test_drop_swapavg', 'epochs_run', 'best_epoch', 'minutes']].mean().round(4).to_string())

In [ ]:
# 6 · Auto-shutdown (releases the GPU)
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()